# HybridGaussianISAM

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridGaussianISAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

`HybridGaussianISAM` implements the Incremental Smoothing and Mapping (ISAM) algorithm for **hybrid** factor graphs, specifically `HybridGaussianFactorGraph`s. It inherits from `gtsam.ISAM<HybridBayesTree>`, meaning it maintains an underlying `HybridBayesTree` representing the smoothed posterior distribution $P(X, M | Z)$ over continuous variables $X$, discrete variables $M$, given measurements $Z$.

The key feature is the `update` method, which efficiently incorporates new factors (measurements) into the existing `HybridBayesTree` without re-processing the entire history. This involves:
1. Identifying the portion of the Bayes tree affected by the new factors.
2. Removing the affected cliques (orphans).
3. Re-eliminating the variables in the orphaned cliques along with the new factors.
4. Merging the newly created Bayes sub-tree back into the main tree.

It provides an incremental solution for problems involving both continuous and discrete variables, where the underlying system dynamics are linear or have been linearized (resulting in a `HybridGaussianFactorGraph`).

In [1]:
import gtsam
import numpy as np

from gtsam import (
    HybridGaussianISAM, HybridGaussianFactorGraph, HybridBayesTree,
    JacobianFactor, DecisionTreeFactor, HybridGaussianFactor,
    DiscreteValues, VectorValues, HybridValues, Ordering
)
from gtsam.symbol_shorthand import X, D

ImportError: cannot import name 'HybridGaussianISAM' from 'gtsam' (c:\Users\porte\miniconda3\envs\gtsam\Lib\site-packages\gtsam\__init__.py)

## Initialization

Can be initialized empty or from an existing `HybridBayesTree`.

In [ ]:
# 1. Empty ISAM
hisam1 = gtsam.HybridGaussianISAM()
print("Empty HybridGaussianISAM created.")

# 2. From existing HybridBayesTree
# Create a minimal initial graph and Bayes tree P(D0), P(X0)
initial_graph = gtsam.HybridGaussianFactorGraph()
dk0 = (D(0), 2)
initial_graph.add(DecisionTreeFactor([dk0], "0.6 0.4")) # P(D0)
initial_graph.add(JacobianFactor(X(0), np.eye(1), np.zeros(1), gtsam.noiseModel.Unit.Create(1))) # P(X0)
ordering = gtsam.Ordering([X(0), D(0)])
initial_hbt = initial_graph.eliminateMultifrontal(ordering)

hisam2 = gtsam.HybridGaussianISAM(initial_hbt)
print("\nHybridGaussianISAM from initial HybridBayesTree:")
hisam2.print()

## Incremental Updates

The `update` method takes a `HybridGaussianFactorGraph` containing new factors to be added.

In [ ]:
# Start with hisam2 from above
hisam = hisam2

# --- Update 1: Add factors connecting X0, X1, D0 ---
update1_graph = gtsam.HybridGaussianFactorGraph()
# Add P(X1 | X0) = N(X0+1, 0.1)
update1_graph.add(JacobianFactor(X(0), -np.eye(1), X(1), np.eye(1), np.array([1.0]), gtsam.noiseModel.Isotropic.Sigma(1, np.sqrt(0.1))))
# Add P(X1 | D0) = mixture N(1, 0.25); N(5, 1.0)
gf0 = JacobianFactor(X(1), np.eye(1), np.array([1.0]), gtsam.noiseModel.Isotropic.Sigma(1, 0.5))
gf1 = JacobianFactor(X(1), np.eye(1), np.array([5.0]), gtsam.noiseModel.Isotropic.Sigma(1, 1.0))
update1_graph.add(HybridGaussianFactor(dk0, [gf0, gf1]))

print("\nAdding Update 1 Factors:")
update1_graph.print()

hisam.update(update1_graph)
print("\nISAM state after Update 1:")
hisam.print()

# --- Update 2: Add factor connecting X1, X2 ---
update2_graph = gtsam.HybridGaussianFactorGraph()
update2_graph.add(JacobianFactor(X(1), -np.eye(1), X(2), np.eye(1), np.array([2.0]), gtsam.noiseModel.Isotropic.Sigma(1, 1.0)))

print("\nAdding Update 2 Factors:")
update2_graph.print()

hisam.update(update2_graph)
print("\nISAM state after Update 2:")
hisam.print()

## Solution and Marginals

After updates, the underlying `HybridBayesTree` can be used to obtain the current MAP estimate or calculate marginals, similar to the batch case.

In [ ]:
# Get the current MAP estimate from the ISAM object
# ISAM inherits optimize() from HybridBayesTree
current_map_solution = hisam.optimize()
print("\nCurrent MAP Solution from ISAM:")
current_map_solution.print()

# Access the underlying HybridBayesTree if needed (though ISAM exposes its methods)
final_hbt = hisam.bayesTree() # Returns a const reference/copy
# log_prob = final_hbt.logProbability(current_map_solution)
# print(f"\nLogProbability at MAP solution: {log_prob}")

# Get a specific GaussianBayesTree for an MPE assignment
mpe = final_hbt.mpe()
print("\nMPE Assignment:", mpe)
gbt_mpe = final_hbt.choose(mpe)
print("\nGaussianBayesTree for MPE assignment:")
gbt_mpe.print()